# Notebook ٨: Embeddings (معنى الكلمة كمتجه)

Co-occurrence وcosine وWord2Vec على نصوص رياضية حقيقية. gensim 4.4، كل خلية اتشغلت فعلًا.

## ١. فكرة الـ co-occurrence

الكلمة معناها من جيرانها. بنعد كل كلمة ظهرت جنب كلمة تانية في نافذة صغيرة.

In [ ]:
# المصفوفات
import numpy as np
# جمل صغيرة
sents = ["salah scored a goal", "haaland scored a goal", "salah assisted a goal", "the referee blew the whistle", "the referee showed a card"]
# المفردات
vocab = sorted({w for s in sents for w in s.split()})
# كلمة ← رقم
ix = {w: i for i, w in enumerate(vocab)}
# مصفوفة co-occurrence فاضية
M = np.zeros((len(vocab), len(vocab)))
# لكل جملة
for s in sents:
# كلماتها
    ws = s.split()
# لكل كلمة
    for i, w in enumerate(ws):
# نافذة ±2
        for j in range(max(0, i - 2), min(len(ws), i + 3)):
# نستثني الكلمة نفسها
            if i != j:
# نزود العدّاد
                M[ix[w], ix[ws[j]]] += 1
# المفردات
print(vocab)
# صف كلمة salah
print(M[ix["salah"]].astype(int))


## ٢. Cosine similarity

بيقيس زاوية بين متجهين، من غير تأثر بطول المتجه (تكرار الكلمة).

In [ ]:
# دالة cosine
def cos(a, b):
# ضرب داخلي ÷ الأطوال
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
# أزواج للمقارنة
for a, b in [("salah", "haaland"), ("salah", "referee"), ("goal", "card")]:
# نطبع التشابه
    print(a, b, round(cos(M[ix[a]], M[ix[b]]), 2))


## ٣. Word2Vec على نصوص رياضية حقيقية

skip-gram: ندرب موديل يتوقع الجيران، ونستخدم الأوزان كـ embeddings.

In [ ]:
# داتا نصوص
from sklearn.datasets import fetch_20newsgroups
# تقطيع وتنضيف بسيط
from gensim.utils import simple_preprocess
# كل المقالات
d = fetch_20newsgroups(subset="all", categories=["rec.sport.baseball", "rec.sport.hockey"], remove=("headers", "footers", "quotes"))
# كل مقال لقايمة كلمات صغيرة
sents = [simple_preprocess(t) for t in d.data]
# عدد المقالات والكلمات
print(len(sents), sum(len(s) for s in sents), "words")
# أول 12 كلمة من مقال
print(sents[5][:12])


In [ ]:
# الموديل
from gensim.models import Word2Vec
# skip-gram بـ negative sampling
w2v = Word2Vec(sents, vector_size=50, window=5, min_count=5, sg=1, negative=5, epochs=15, workers=1, seed=0)
# حجم الـ vocab وأبعاد المتجه
print(len(w2v.wv), "words,", w2v.wv.vector_size, "dims")
# أول 5 أرقام من متجه hockey
print(w2v.wv["hockey"][:5].round(2))


In [ ]:
# كلمات للتجربة
for w in ["hockey", "pitcher", "goalie", "playoffs"]:
# أقرب 5 كلمات لكل واحدة
    print(w, "->", [x for x, _ in w2v.wv.most_similar(w, topn=5)])


In [ ]:
# اختصار للمتجهات
wv = w2v.wv
# تشابه كلمتين من نفس الرياضة
print(round(wv.similarity("pitcher", "batter"), 2))
# تشابه كلمتين من رياضتين مختلفتين
print(round(wv.similarity("pitcher", "goalie"), 2))


## ٤. Embeddings للجملة كلها

أبسط طريقة: متوسط متجهات الكلمات. ونستخدمها في تصنيف.

In [ ]:
# تقسيم
from sklearn.model_selection import train_test_split
# الموديل
from sklearn.linear_model import LogisticRegression
# متجه الجملة
def sent_vec(s):
# متجهات الكلمات المعروفة
    v = [wv[w] for w in s if w in wv]
# المتوسط أو أصفار
    return np.mean(v, axis=0) if v else np.zeros(50)
# متجه لكل مقال
X = np.array([sent_vec(s) for s in sents])
# الـ labels
y = d.target
# تقسيم
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)
# تدريب
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
# الدقة بمتجهات 50 بعد بس
print("accuracy with 50-d embeddings:", round(clf.score(Xte, yte), 3))


ملحوظة: الـ embeddings دي static: الكلمة ليها متجه واحد دايمًا. BERT (في الـ notebooks الجاية) بيدي متجه مختلف حسب الجملة.